# Preprocesado y Feature Engineering

En este notebook transformamos el dataset saneado (`data_sanitized.csv`) de transacciones individuales en una **serie temporal diaria** lista para ser modelizada, enriqueciéndola mediante ingeniería de variables.

El notebook es **autosuficiente**: carga el CSV desde disco sin depender de ningún notebook previo.

Las etapas que se realizan son:
- **4.1** Agregación diaria y creación de la variable objetivo `Ventas`
- **4.2** Variables categóricas de resumen diario (producto top, país top, clientes únicos)
- **4.3** Extracción de variables temporales desde la fecha
- **4.4** Análisis y limpieza de las nuevas variables
- **4.6** Análisis de correlación y selección de variables
- **4.7** Lags (retrasos temporales)
- **4.8** Medias móviles
- **4.9** Eventos especiales
- **4.10** Escalado y exportación (train/test split + CSV)


## 0. Imports y Carga del Dataset

Preparamos el entorno de trabajo: importamos librerías, configuramos la estética visual y definimos las rutas del proyecto.

A continuación cargamos `data_sanitized.csv` — el CSV limpio exportado por el notebook de limpieza — que ya incluye las columnas auxiliares `Fecha`, `Mes`, `DiaSemana`, `EsCancelacion` y `TotalPrice`, por lo que no es necesario recalcularlas.

Se inicializa también el diccionario de auditoría `stats_preprocesing` que registrará las transformaciones realizadas en este notebook.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Configuración visual global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
%matplotlib inline

# Rutas del proyecto
RUTA_INTERIM   = '../../../data/interim/'
RUTA_GRAFICOS  = '../../../graphics/'
RUTA_PROCESSED = '../../../data/processed/'
os.makedirs(RUTA_GRAFICOS,  exist_ok=True)
os.makedirs(RUTA_PROCESSED, exist_ok=True)

print('Librerías cargadas correctamente.')


In [ ]:
# Carga del CSV limpio (output del notebook de limpieza)
df_raw     = pd.read_csv(
    RUTA_INTERIM + 'data_sanitized.csv',
    parse_dates=['InvoiceDate', 'Fecha']
)
df_working = df_raw.copy()

# EsCancelacion puede llegar como string 'True'/'False' → forzar a bool
df_working['EsCancelacion'] = df_working['EsCancelacion'].astype(bool)

# Diccionario de auditoría (se actualiza en cada paso de transformación)
stats_preprocesing = {
    'Registros de entrada (transacciones)': len(df_raw),
}

# ── Resumen de carga ──────────────────────────────────────────────────────────
print('=' * 58)
print(f'  DATASET CARGADO  (data_sanitized.csv)')
print('=' * 58)
print(f'  Filas    : {df_raw.shape[0]:>10,}')
print(f'  Columnas : {df_raw.shape[1]:>10}')
print(f'  Rango    : {df_raw["Fecha"].min().date()} → {df_raw["Fecha"].max().date()}')
print('=' * 58)
print(f'\n  df_working activo : {len(df_working):,} filas')
print(f'  Columnas heredadas del notebook de limpieza:')
print(f'    - Fecha, Mes, DiaSemana, EsCancelacion, TotalPrice')
print(f'\n  Columnas completas: {list(df_working.columns)}')


In [ ]:
# [DOC] generamos variables temporales básicas, variables de memoria y ventanas móviles

# [DOC] Variables Temporales Básicas
df_daily_sales['DayOfWeek'] = df_daily_sales['Date'].dt.dayofweek
df_daily_sales['Month'] = df_daily_sales['Date'].dt.month
df_daily_sales['IsWeekend'] = df_daily_sales['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# [DOC] Variables de Memoria (Lags)
# No solo el día anterior (Lag 1), sino el mismo día de la semana pasada (Lag 7)
df_daily_sales['Sales_Lag1'] = df_daily_sales['Sales'].shift(1)
df_daily_sales['Sales_Lag7'] = df_daily_sales['Sales'].shift(7)

# [DOC] Ventanas Móviles (Rolling Windows)
# Promedio de ventas de los últimos 7 días (suaviza picos aislados)
df_daily_sales['Rolling_Mean_7'] = df_daily_sales['Sales'].shift(1).rolling(window=7).mean()

df_daily_sales.dropna(inplace=True)

In [ ]:
# [DOC] Aplicamos One-Hot Encoding para el día de la semana y separamos el conjunto de entrenamiento y test antes de escalar

df_daily_sales = pd.get_dummies(df_daily_sales, columns=['DayOfWeek'], prefix='Day')

split_date = pd.to_datetime('2011-11-09')

df_train = df_daily_sales[df_daily_sales['Date'] < split_date].copy()
df_test = df_daily_sales[df_daily_sales['Date'] >= split_date].copy()

print(f"[INFO] Registros Entrenamiento: {len(df_train)}")
print(f"[INFO] Registros Test: {len(df_test)}")

[INFO] Registros Entrenamiento: 271
[INFO] Registros Test: 27


In [10]:
# [DOC] Escalamos las variables numéricas para que todas tengan el mismo peso en el modelo.

features_to_scale = ['Sales_Lag1', 'Sales_Lag7', 'Rolling_Mean_7', 'TransactionCount']
scaler = StandardScaler()

df_train[features_to_scale] = scaler.fit_transform(df_train[features_to_scale])
df_test[features_to_scale] = scaler.transform(df_test[features_to_scale])

In [11]:
# [DOC] Realizamos el guardado de los datos procesados

output_dir = '../../data/processed'

if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)
    print(f"[INFO] Carpeta creada con éxito: {output_dir}")

df_train.to_csv(os.path.join(output_dir, 'train_data.csv'), index=False)
df_test.to_csv(os.path.join(output_dir, 'test_data.csv'), index=False)

print(f"[SUCCESS] Archivos guardados correctamente en: {output_dir}")
display(df_train.head())

[INFO] Carpeta creada con éxito: ../../data/processed
[SUCCESS] Archivos guardados correctamente en: ../../data/processed


,Date,Sales,TransactionCount,CustomerID,Month,IsWeekend,Sales_Lag1,Sales_Lag7,Rolling_Mean_7,Day_0,Day_1,Day_2,Day_3,Day_4,Day_6
7,2010-12-09,30218.95,2.186918,88,12,0,1.527858,1.701851,1.523415,False,False,False,True,False,False
8,2010-12-10,27177.24,0.911706,53,12,0,1.031841,1.916561,1.388964,False,False,False,False,True,False
9,2010-12-12,16788.64,-0.629175,41,12,1,0.693646,0.145325,1.136129,False,False,False,False,False,True
10,2010-12-13,24829.71,0.486635,58,12,0,-0.461420,0.924451,1.008887,True,False,False,False,False,False
11,2010-12-14,26671.93,1.283643,71,12,0,0.432633,0.610659,0.909659,False,True,False,False,False,False
